# XAUUSD PPO Training — Google Colab / Kaggle

Train baseline PPO on Gold Futures **daily** data (no yfinance limit).

**Runtime**: GPU T4 → Runtime → Change runtime type → GPU  
**Time**: ~15-25 min on T4 | ~60-90 min on CPU

In [ ]:
!pip install -q 'stable-baselines3[extra]>=2.3.0' 'gymnasium>=0.29' 'yfinance>=0.2.40' 'ta>=0.11' matplotlib

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import yfinance as yf
import ta
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback
import torch, os, matplotlib.pyplot as plt

print(f'numpy  {np.__version__}')
print(f'torch  {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')
print('✓ imports ok')

In [ ]:
# ── Config ───────────────────────────────────────────────────────────────────
SYMBOL      = 'GC=F'          # Gold Futures
INTERVAL    = '1d'            # daily — no yfinance limit, stable training
TRAIN_START = '2018-01-01'
TRAIN_END   = '2024-06-01'    # ~6.5 years = ~1600 daily bars
TEST_START  = '2024-06-01'
TEST_END    = '2025-06-01'    # 1 year out-of-sample
TIMESTEPS   = 1_000_000
OUTPUT      = 'baseline_ppo'

In [ ]:
# ── Data loader ───────────────────────────────────────────────────────────────
def load_data(start: str, end: str) -> pd.DataFrame:
    print(f'Downloading {SYMBOL} {start} → {end} ({INTERVAL}) ...')
    raw = yf.download(SYMBOL, start=start, end=end, interval=INTERVAL,
                      auto_adjust=True, progress=False)
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.get_level_values(0)
    raw.columns = [c.lower() for c in raw.columns]
    df = raw[['open', 'high', 'low', 'close', 'volume']].dropna().copy()
    df = _add_indicators(df)
    print(f'  → {len(df)} bars  |  {df.index[0].date()} → {df.index[-1].date()}')
    return df

def _add_indicators(df: pd.DataFrame) -> pd.DataFrame:
    c, h, l = df['close'], df['high'], df['low']
    df = df.copy()
    df['rsi']         = ta.momentum.RSIIndicator(c, 14).rsi()
    macd              = ta.trend.MACD(c, 12, 26, 9)
    df['macd']        = macd.macd()
    df['macd_signal'] = macd.macd_signal()
    df['macd_diff']   = macd.macd_diff()
    bb                = ta.volatility.BollingerBands(c, 20, 2)
    df['bb_pct']      = bb.bollinger_pband()
    df['ema50']       = ta.trend.EMAIndicator(c, 50).ema_indicator()
    df['ema200']      = ta.trend.EMAIndicator(c, 200).ema_indicator()
    df['atr']         = ta.volatility.AverageTrueRange(h, l, c, 14).average_true_range()
    df['close_pct']   = c.pct_change()
    df['hl_ratio']    = (h - l) / c
    return df.dropna()

df_train = load_data(TRAIN_START, TRAIN_END)
df_test  = load_data(TEST_START,  TEST_END)
print(f'\nTrain: {len(df_train)} bars | Test: {len(df_test)} bars')
df_train.tail(3)

In [ ]:
# ── Gym Environment ───────────────────────────────────────────────────────────
class XAUUSDEnv(gym.Env):
    metadata = {'render_modes': []}
    TC   = 0.0002   # transaction cost per unit position change
    DD_P = 0.10     # drawdown penalty coefficient

    def __init__(self, df: pd.DataFrame, initial_cash: float = 100_000.0):
        super().__init__()
        self.df           = df.reset_index(drop=True)
        self.initial_cash = initial_cash
        self.n            = len(df)
        self.observation_space = spaces.Box(-np.inf, np.inf, (20,), dtype=np.float32)
        self.action_space      = spaces.Box(-1.0, 1.0, (1,), dtype=np.float32)
        self._reset_state()

    def _reset_state(self):
        self.i    = 0
        self.pos  = 0.0
        self.nav  = self.initial_cash
        self.peak = self.initial_cash
        self.bars = 0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self._reset_state()
        return self._obs(), {}

    def step(self, action):
        new_pos = float(np.clip(action[0], -1.0, 1.0))
        row     = self.df.iloc[self.i]
        price   = float(row['close'])

        # PnL from holding previous position
        pnl = 0.0
        if self.i > 0:
            prev = float(self.df.iloc[self.i - 1]['close'])
            pnl  = self.pos * (price - prev) / prev * self.nav

        cost      = abs(new_pos - self.pos) * self.TC * self.initial_cash
        self.pos  = new_pos
        self.nav  = max(self.nav + pnl - cost, 1.0)
        self.peak = max(self.peak, self.nav)
        dd        = (self.peak - self.nav) / self.peak

        reward = (pnl - cost) / self.initial_cash
        if dd > 0.02:
            reward -= self.DD_P * dd

        self.i    += 1
        self.bars  = self.bars + 1 if self.pos != 0 else 0
        terminated = self.i >= self.n - 1
        obs        = self._obs() if not terminated else np.zeros(20, dtype=np.float32)
        return obs, float(reward), terminated, False, {'nav': self.nav, 'dd': dd}

    def _obs(self) -> np.ndarray:
        row   = self.df.iloc[self.i]
        price = float(row['close'])
        p     = price + 1e-8

        def s(col):
            v = row.get(col, 0.0)
            return float(np.nan_to_num(float(v), nan=0.0, posinf=0.0, neginf=0.0))

        # day-of-week cycle (Monday=0 … Friday=4)
        dow = self.df.index[self.i] if hasattr(self.df.index, 'dayofweek') else self.i
        try:
            dow_norm = pd.Timestamp(self.df.index[self.i]).dayofweek / 4.0
        except Exception:
            dow_norm = (self.i % 5) / 4.0

        obs = np.array([
            s('close_pct'),
            s('hl_ratio'),
            s('rsi') / 100.0,
            s('macd') / (p * 0.01),
            s('macd_diff') / (p * 0.01),
            s('bb_pct'),
            (s('ema50')  - price) / p,
            (s('ema200') - price) / p,
            s('atr') / p,
            dow_norm,
            0.0, 0.5, 0.5,   # LLM signal placeholders (filled at inference)
            self.pos,
            (self.nav  - self.initial_cash) / self.initial_cash,
            (self.peak - self.nav) / self.peak,
            float(np.minimum(self.bars, 100)) / 100.0,
            0.0, 0.0, 0.0,
        ], dtype=np.float32)
        return np.clip(obs, -10.0, 10.0)

# Sanity check
env_check = XAUUSDEnv(df_train)
obs, _ = env_check.reset()
print(f'obs shape: {obs.shape}  |  all finite: {np.all(np.isfinite(obs))}')
obs2, r, done, _, info = env_check.step(np.array([0.5]))
print(f'step ok  |  reward: {r:.6f}  |  nav: {info["nav"]:,.0f}')

In [ ]:
# ── Train PPO ─────────────────────────────────────────────────────────────────
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
os.makedirs('models', exist_ok=True)

train_env = make_vec_env(lambda: XAUUSDEnv(df_train), n_envs=4)
eval_env  = make_vec_env(lambda: XAUUSDEnv(df_train), n_envs=1)

model = PPO(
    'MlpPolicy', train_env,
    verbose=1,
    device=device,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=256,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,
)

callbacks = [
    EvalCallback(
        eval_env,
        best_model_save_path='models/',
        eval_freq=max(50_000 // 4, 1),   # divide by n_envs
        n_eval_episodes=3,
        deterministic=True,
        verbose=1,
    ),
    CheckpointCallback(save_freq=max(100_000 // 4, 1), save_path='models/',
                       name_prefix='ppo_ckpt'),
]

print(f'Training {TIMESTEPS:,} timesteps on {len(df_train)} bars...')
model.learn(total_timesteps=TIMESTEPS, callback=callbacks, progress_bar=True)
model.save(OUTPUT)
print(f'\n✓ Saved: {OUTPUT}.zip')

In [ ]:
# ── Backtest out-of-sample ────────────────────────────────────────────────────
def backtest(model, df):
    env  = XAUUSDEnv(df)
    obs, _ = env.reset()
    navs = [env.initial_cash]
    done = False
    while not done:
        act, _ = model.predict(obs, deterministic=True)
        obs, _, terminated, truncated, info = env.step(act)
        navs.append(info['nav'])
        done = terminated or truncated
    navs = np.array(navs, dtype=np.float64)
    rets = np.diff(navs) / navs[:-1]
    sharpe   = float((rets.mean() / (rets.std() + 1e-10)) * np.sqrt(252))  # daily → annual
    total_r  = float((navs[-1] / navs[0] - 1) * 100)
    running_max = np.maximum.accumulate(navs)
    max_dd   = float(((running_max - navs) / running_max).max() * 100)
    win_rate = float((rets > 0).sum() / len(rets) * 100)
    return navs, {'sharpe': sharpe, 'return': total_r, 'max_dd': max_dd, 'win_rate': win_rate}

best_model = PPO.load('models/best_model') if os.path.exists('models/best_model.zip') else PPO.load(OUTPUT)
navs, m = backtest(best_model, df_test)

print('\n=== Out-of-sample Backtest ===')
print(f'  Period       : {TEST_START} → {TEST_END}')
print(f'  Total Return : {m["return"]:+.2f}%')
print(f'  Sharpe Ratio : {m["sharpe"]:.3f}   (target > 0.5)')
print(f'  Max Drawdown : {m["max_dd"]:.2f}%')
print(f'  Win Rate     : {m["win_rate"]:.1f}%')
print(f'  Final NAV    : ${navs[-1]:,.0f}')

if m['sharpe'] > 0.5:
    print('\n✓ Sharpe > 0.5 — model ready to deploy')
else:
    print('\n⚠ Sharpe < 0.5 — consider more timesteps or tuning reward')

In [ ]:
# ── Equity curve ─────────────────────────────────────────────────────────────
# Buy-and-hold baseline
bh = df_test['close'].values
bh_nav = 100_000 * bh / bh[0]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 7), gridspec_kw={'height_ratios': [3, 1]})

ax1.plot(navs,   color='#00CC96', lw=1.5, label='PPO Policy')
ax1.plot(bh_nav, color='#636EFA', lw=1,   linestyle='--', label='Buy & Hold')
ax1.axhline(100_000, color='gray', lw=0.8, linestyle=':')
ax1.set_title(f'XAUUSD PPO — Out-of-sample {TEST_START}→{TEST_END}  |  Sharpe={m["sharpe"]:.2f}')
ax1.set_ylabel('Portfolio Value ($)')
ax1.legend()

# Drawdown
running_max = np.maximum.accumulate(navs)
dd_series   = (running_max - navs) / running_max * 100
ax2.fill_between(range(len(dd_series)), -dd_series, 0, color='#EF553B', alpha=0.5)
ax2.set_ylabel('Drawdown %')
ax2.set_xlabel('Days')

plt.tight_layout()
plt.savefig('equity_curve.png', dpi=130)
plt.show()
print('Saved equity_curve.png')

In [ ]:
# ── Download model ────────────────────────────────────────────────────────────
import shutil

# Copy best_model if it exists, else use baseline
src = 'models/best_model.zip' if os.path.exists('models/best_model.zip') else f'{OUTPUT}.zip'
shutil.copy(src, 'baseline_ppo.zip')
print(f'Packed: {src} → baseline_ppo.zip')

try:
    from google.colab import files
    files.download('baseline_ppo.zip')
    print('\nColab: download started')
    print('→ Lưu file vào  models/baseline_ppo.zip  trong repo local')
except ImportError:
    shutil.copy('baseline_ppo.zip', '/kaggle/working/baseline_ppo.zip')
    print('\nKaggle: → Output tab → Download baseline_ppo.zip')
    print('→ Lưu file vào  models/baseline_ppo.zip  trong repo local')